# FatFormer XLA — audit đầy đủ danh mục dữ liệu và đánh giá model thật

**Bản sửa 15/11/2026. Chưa chạy toàn bộ trên Colab trong phiên sửa notebook.**

Notebook đọc và giải mã từng ảnh, kiểm hash/nhãn/split rồi chạy `package.predict()` thật. Không dùng bảng metric chép sẵn, không thay ảnh hỏng bằng tensor, không fit lại calibration và không huấn luyện. Kết quả train/validation/calibration là chẩn đoán riêng, không gộp vào test.

Phạm vi bắt buộc: ProGAN train/val; diffusion staging và manifest legacy; data v2 (12.240 train, 1.080 val_select, 1.080 val_calib); CFE frozen 400 calibration/1.600 evaluation; 18 tập chuẩn + 5 tập bổ sung; toàn bộ 6 biến thể trong `test_degraded.tar`. Các con số là số bản ghi/lượt ảnh, không phải nội dung ảnh độc lập. 18 tập + supplemental được đánh giá ở 7 điều kiện sau crop; archive degraded cũ được đọc nguyên biến thể, không nén lần hai.

Source lấy trực tiếp từ GitHub, nhánh `hoang`, ở cell 2. Mặc định chốt commit của nhánh tại lúc bắt đầu; để tái lập một run, đặt `GIT_COMMIT` bằng SHA 40 ký tự đã ghi trong `git_source.json`. Checkout detached và kiểm source sạch trước khi chạy. Frozen CFE splits dạng JSON đã có trong Git; không cần ZIP source hay upload NPZ. Weights và các archive ảnh vẫn đọc từ Drive.

Chọn GPU Colab. Phải có đủ cả 8 archive trong cấu hình; nếu thiếu, notebook ghi thất bại và dừng. Không tạo tuyên bố hoàn tất từ dữ liệu thiếu. Đây là lượt đo lớn, có thể mất nhiều giờ; chưa có ước lượng runtime toàn bộ đã kiểm chứng. Dữ liệu ảnh chỉ giải nén từng archive trên SSD và được dọn sau mỗi archive.


In [ ]:
# 1. Drive, run mới và lưu bằng chứng cả khi có lỗi.
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
import importlib.util, json, os, shutil, subprocess, sys, tempfile, uuid

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(DRIVE_ROOT)
RUN_ID = 'master_audit_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
RUN_ROOT = Path('/content')/RUN_ID
REPO = RUN_ROOT/'code'
OUTPUT = RUN_ROOT/'results'
STAGING = RUN_ROOT/'staging'
for folder in (REPO, OUTPUT, STAGING):
    folder.mkdir(parents=True, exist_ok=False)
BATCH_SIZE = 16  # Không tự đổi batch giữa chừng. OOM: run mới, batch nhỏ hơn, ghi khác protocol.
BOOTSTRAP = 1000
FULL_REPOSITORY_TESTS = False  # Mặc định chạy nhóm test liên quan, không gọi là full-suite.
state = dict(run_id=RUN_ID, execution_status='configured', steps=[],
    scientific_status='post_hoc_remeasurement; not a new blind test',
    training=False, calibration_refit=False, batch_size=BATCH_SIZE,
    bootstrap=BOOTSTRAP, full_repository_tests_requested=FULL_REPOSITORY_TESTS)

def persist(status):
    state['execution_status'] = status
    (OUTPUT/'run_status.json').write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    package = Path(shutil.make_archive(str(RUN_ROOT/RUN_ID), 'zip', OUTPUT))
    destination = DRIVE_ROOT/'audit_reports'/RUN_ID
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(package, destination/package.name)
    print('Evidence ZIP:', destination/package.name, '| status:', status)

@contextmanager
def step(name):
    entry = dict(name=name, status='running')
    state['steps'].append(entry)
    try:
        yield
    except Exception as error:
        entry.update(status='failed', error_type=type(error).__name__, error=str(error))
        persist('failed')
        raise
    else:
        entry['status'] = 'completed'
        (OUTPUT/'run_status.json').write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')

print('Run:', RUN_ID)


In [ ]:
# 2. Clone Git và chốt commit source; không dùng ZIP hoặc bảng kết quả cũ.
GIT_REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'
GIT_BRANCH = 'hoang'
GIT_COMMIT = ''  # Đặt SHA đầy đủ để tái lập; rỗng = chốt tip của hoang tại lúc bắt đầu.
SOURCE_PATHS = ['src', 'tools', 'tests', 'configs', 'DATASET/manifests',
                'notebooks/master_full_data_audit_colab.ipynb']
with step('git_source'):
    if GIT_COMMIT and (len(GIT_COMMIT) != 40 or any(c not in '0123456789abcdefABCDEF' for c in GIT_COMMIT)):
        raise ValueError('GIT_COMMIT phải là SHA commit 40 ký tự hoặc để rỗng.')
    git_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    subprocess.run(['git', 'clone', '--filter=blob:none', '--single-branch', '--branch', GIT_BRANCH,
                    GIT_REPO_URL, str(REPO)], env=git_env, check=True)
    def git(*arguments):
        return subprocess.check_output(['git', *arguments], cwd=REPO, env=git_env, text=True).strip()
    commit = git('rev-parse', '--verify', (GIT_COMMIT or 'refs/remotes/origin/'+GIT_BRANCH)+'^{commit}')
    subprocess.run(['git', 'merge-base', '--is-ancestor', commit, 'refs/remotes/origin/'+GIT_BRANCH],
                   cwd=REPO, env=git_env, check=True)
    subprocess.run(['git', 'checkout', '--detach', commit], cwd=REPO, env=git_env, check=True)
    required = ['tools/master_data_audit.py', 'tools/phase7_git_source.py', 'tests/test_master_data_audit.py',
                'configs/master_audit/calibration_split.json', 'configs/master_audit/evaluation_split.json',
                'DATASET/manifests/data_manifest_v2.json', 'DATASET/manifests/data_manifest.json']
    for name in required:
        if not (REPO/name).is_file():
            raise FileNotFoundError(f'Commit {commit} thiếu {name}. Dừng, không dùng code cũ.')
        git('ls-files', '--error-unmatch', '--', name)
    os.chdir(REPO)
    sys.path.insert(0, str(REPO))
    from tools.phase7_git_source import git_source_identity
    source = git_source_identity(REPO, source_paths=SOURCE_PATHS)
    if source['commit'] != commit:
        raise ValueError('Source checkout khác commit đã chọn.')
    source['repository_url'] = GIT_REPO_URL
    state['git_source'] = source
    (OUTPUT/'git_source.json').write_text(json.dumps(source, indent=2), encoding='utf-8')
    print('Git commit đã chốt:', commit)


In [ ]:
# 3. Giữ torch/torchvision runtime, chỉ cài dependency phụ còn thiếu; lưu môi trường thực tế.
with step('environment_and_tests'):
    dependencies = {'pytest':'pytest', 'yaml':'PyYAML', 'sklearn':'scikit-learn',
        'matplotlib':'matplotlib', 'regex':'regex', 'tqdm':'tqdm', 'scipy':'scipy'}
    missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    import torch, torchvision, numpy as np
    from tools.master_data_audit import (PACKAGE_SHA, SOURCE_SHA, catalog, manifest_maps, open_inventory,
        audit_archive, check_overlap, evaluate_archive, final_metrics, jobs_for, export_summary_csv)
    from tools.phase7_corrected import write_json
    from tools.calibration_data import sha256_file
    from tools.infer_unified import load_unified_model, benchmark_latency
    if not torch.cuda.is_available():
        raise RuntimeError('Cần runtime GPU. Không tự chuyển CPU cho lượt benchmark này.')
    if BATCH_SIZE < 1 or BOOTSTRAP < 2:
        raise ValueError('Batch phải dương; bootstrap ít nhất 2.')
    torch.set_num_threads(4)
    torch.manual_seed(42)
    np.random.seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    state['environment'] = dict(python=sys.version, torch=str(torch.__version__), torchvision=str(torchvision.__version__),
        gpu=torch.cuda.get_device_name(0), precision='FP16 backbone autocast; FP32 logits/softmax', tf32=False)
    (OUTPUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
    test_files = ['tests/test_master_data_audit.py', 'tests/test_canonical_loader.py',
        'tests/test_calibration_partitions.py', 'tests/test_temperature_convergence.py',
        'tests/test_phase7_corrected.py', 'tests/test_phase7_git_source.py', 'tests/test_phase7_acceptance.py', 'tests/test_router_pipeline.py',
        'tests/test_loss_v2.py', 'tests/test_selection_metric.py', 'tests/test_selection_regressions.py', 'tests/test_degradation_v2.py']
    command = [sys.executable, '-m', 'pytest', *(['tests'] if FULL_REPOSITORY_TESTS else test_files),
               '-q', '-p', 'no:cacheprovider', '--junitxml', str(OUTPUT/'tests.xml')]
    with (OUTPUT/'tests_console.txt').open('w', encoding='utf-8') as handle:
        outcome = subprocess.run(command, cwd=REPO, stdout=handle, stderr=subprocess.STDOUT, text=True)
    state['tests'] = dict(command=command, exit_code=outcome.returncode,
                         scope='full repository' if FULL_REPOSITORY_TESTS else 'targeted related files; not full repository')
    print((OUTPUT/'tests_console.txt').read_text(encoding='utf-8')[-4000:])
    if outcome.returncode:
        raise RuntimeError('Tests không đạt. Dừng; xem tests.xml và console, không bỏ qua để chạy tiếp.')


In [ ]:
# 4. Tiền kiểm đúng cả danh mục. Sửa đường dẫn ở đây nếu Drive đã được tổ chức khác.
with step('catalog_preflight'):
    ARCHIVES = catalog(DRIVE_ROOT)
    PACKAGE_DRIVE = DRIVE_ROOT/'checkpoint/phase7/corrected_20261109_102954_3d28ad/unified_fatformer_v2.pth'
    missing = [s['path'] for s in ARCHIVES if not Path(s['path']).is_file()]
    if not PACKAGE_DRIVE.is_file():
        missing.append(str(PACKAGE_DRIVE))
    write_json(OUTPUT/'catalog_preflight.json', dict(archives=ARCHIVES, missing=missing))
    if missing:
        raise FileNotFoundError('Thiếu các đầu vào bắt buộc:' + chr(10) + chr(10).join(missing))
    maps = manifest_maps(REPO)
    state['input_manifest_hashes'] = {str(p.relative_to(REPO)):sha256_file(p)
        for folder, pattern in [('DATASET/manifests','*.json'), ('configs/master_audit','*.json')]
        for p in (REPO/folder).glob(pattern)}
    db = open_inventory(OUTPUT/'image_inventory.sqlite')
    print('Đủ đường dẫn 8 archives; chưa đồng nghĩa nội dung đã được audit.')


In [ ]:
# 5. Audit toàn bộ ảnh, không giải nén toàn kho. Lỗi ảnh/hash/nhãn/count sẽ dừng.
with step('all_archive_content_audit'):
    audits = []
    for spec in ARCHIVES:
        report = audit_archive(spec, db, maps, OUTPUT/(spec['name']+'_audit.json'))
        spec['audited_sha256'] = report['archive_sha256']
        audits.append(report)
    state['archive_image_records'] = sum(row['images'] for row in audits)
    state['distinct_encoded_file_hashes'] = db.execute('SELECT count(DISTINCT sha256) FROM images').fetchone()[0]
    write_json(OUTPUT/'archive_audit_summary.json', dict(archives=audits,
        total_records=state['archive_image_records'], unique_encoded_hashes=state['distinct_encoded_file_hashes'],
        note='Archives/splits/replay/degraded views can overlap. Counts do not establish statistical independence.'))
    print('Đã audit bản ghi ảnh:', state['archive_image_records'])


In [ ]:
# 6. Exact-content leakage trước inference. Không giả đã kiểm pHash/generator disjoint toàn kho.
with step('content_overlap'):
    overlap = check_overlap(db, OUTPUT/'content_overlap.json')
    state['perceptual_leakage_audit'] = 'not_run; exact SHA256 only'
    print('Không phát hiện exact-content leakage trong các đối chiếu đã định. pHash chưa chạy.')


In [ ]:
# 7. Nạp đúng package qua canonical loader; không coi state_dict là nn.Module.
with step('model_load'):
    current_source = git_source_identity(REPO, source_paths=SOURCE_PATHS)
    if current_source['commit'] != state['git_source']['commit'] or current_source['files'] != state['git_source']['files']:
        raise ValueError('Source/input thay đổi sau bước kiểm tra Git.')
    package = RUN_ROOT/'unified_fatformer_v2.pth'
    shutil.copy2(PACKAGE_DRIVE, package)
    if sha256_file(package) != PACKAGE_SHA:
        raise ValueError('Package không khớp model đã chọn.')
    device = torch.device('cuda')
    model, metadata = load_unified_model(str(package), clip_path='', device='cuda')
    if metadata['source_checkpoint_sha256'] != SOURCE_SHA or metadata['layout']['indices'] != [7,15,23] or model.router_enabled:
        raise ValueError('Sai source/layout/router.')
    if model.clean_temperature != 2.7832491397857666 or model.fixed_threshold != 0.7918508648872375:
        raise ValueError('Calibration trong package khác bản đã chọn.')
    state['package_sha256'] = PACKAGE_SHA
    state['source_checkpoint_sha256'] = SOURCE_SHA
    state['temperature'] = model.clean_temperature
    state['threshold'] = model.fixed_threshold
    state['protocol_comparison'] = ('same FP16/batch16/TF32-off contract' if BATCH_SIZE == 16
                                    else 'different batch from historical GPU benchmark; report separately')
    write_json(OUTPUT/'package_metadata.json', metadata)


In [ ]:
# 8. Chạy model thật trên toàn bộ archive; train/val/calibration là diagnostic riêng.
# Không giới hạn sample. Chỉ một archive giải nén mỗi lần; tự dọn staging khi kết thúc hoặc lỗi.
with step('all_dataset_predictions'):
    reports = []
    expected_jobs = set()
    for spec in ARCHIVES:
        partitions = [dict(split=row[0]) for row in db.execute('SELECT DISTINCT split FROM images WHERE archive=?', (spec['name'],))]
        expected_jobs.update((spec['name'], split, condition) for split, condition, _ in jobs_for(spec, partitions))
    for spec in ARCHIVES:
        with tempfile.TemporaryDirectory(prefix=spec['name']+'_', dir=STAGING) as temporary:
            rows = evaluate_archive(spec, db, Path(temporary), OUTPUT/'predictions', model, device,
                                    batch_size=BATCH_SIZE, bootstrap=BOOTSTRAP)
            reports.extend(rows)
            write_json(OUTPUT/'job_reports.json', reports)
        print('Xong archive:', spec['name'])
    if len(reports) != len(expected_jobs) or {(r['archive'],r['split'],r['condition']) for r in reports} != expected_jobs:
        raise ValueError('Job coverage thiếu hoặc trùng. Không công bố full run.')
    state['expected_jobs'] = len(expected_jobs)
    state['completed_jobs'] = len(reports)
    state['measured_prediction_records'] = sum(r['pooled']['total_samples'] for r in reports)
    summary = final_metrics(reports, OUTPUT/'predictions', model.fixed_threshold, BOOTSTRAP)
    write_json(OUTPUT/'benchmark_summary.json', summary)
    export_summary_csv(summary, OUTPUT/'measured_condition_and_subset_summary.csv')
    print('Lượt dự đoán đo mới:', state['measured_prediction_records'])


In [ ]:
# 9. Latency synthetic riêng: paired backbone/package, warmup20, steps200, median/p95/overhead.
with step('latency_and_gates'):
    latency = benchmark_latency(model, device, batch_sizes=[1,8,16,32], n_warmup=20, n_steps=200,
        output_json=str(OUTPUT/'latency.json'), checkpoint_path=str(package))
    gates = {'G7.0': dict(status='NOT_REEVALUATED', reason='Legacy loader/CI DoD needs a separate audit; tests here do not close it.'),
        'G7.1': dict(status='NOT_REEVALUATED', reason='Historical Gen2 comparison is not a new measurement of this package.'),
        'G7.2': summary['G72'],
        'G7.3': dict(status='PENDING', measured_gpu_overhead=latency['overhead_meets_target'],
            single_backbone_forward_checked=True, best_by_condition_comparison='NOT_RUN')}
    write_json(OUTPUT/'gates.json', gates)
    print(json.dumps(gates, indent=2, ensure_ascii=False))


In [ ]:
# 10. Bàn giao chỉ bằng chứng của run này; không ghi đè run cũ hoặc weights/calibration.
with step('finalize'):
    required = {'git_source', 'environment_and_tests', 'catalog_preflight', 'all_archive_content_audit',
                'content_overlap', 'model_load', 'all_dataset_predictions', 'latency_and_gates'}
    previous = state['steps'][:-1]
    if not required.issubset({s['name'] for s in previous}) or any(s['status'] != 'completed' for s in previous):
        raise RuntimeError('Không thể finalize khi thiếu bước hoặc có bước thất bại. Tạo run mới.')
    if state.get('completed_jobs') != state.get('expected_jobs') or not state.get('completed_jobs'):
        raise RuntimeError('Chưa có đủ job đo thực tế.')
    db.commit()
    db.close()
    state['coverage'] = 'all eight configured archives audited and evaluated; no image sampling'
    state['interpretation'] = 'execution completion is not a PASS of all scientific/product gates'
    state['limits'] = ['post-hoc remeasurement, not blind test',
        'exact-content overlap checks only; perceptual/generator-disjoint audit not completed',
        'targeted tests only unless full_repository_tests_requested=true',
        'no localization ground truth, no forensic reliability certification',
        'CI by image does not cover all repeated-reference/generator dependencies']
persist('completed')
print('Đã hoàn tất thực thi theo phạm vi. Đọc gates.json và run_status.json; không mặc định mọi gate PASS.')


## Đọc kết quả

`archive_audit_summary.json` và SQLite là coverage/ảnh lỗi/hash/nhãn; `job_reports.json` tách từng archive/split/condition; `predictions/*.npz` chứa dự đoán từng ảnh và source/package hash. `benchmark_summary.json` tách CFE và core18 theo condition, không trộn 5 supplemental hay train vào test. `latency.json` đo tensor tổng hợp, không gồm I/O. `gates.json` tính G7.2 từ run mới, giữ G7.3 pending khi thiếu best-reference.

Nếu một bước lỗi, ZIP có `execution_status=failed` và không phải bằng chứng full-run thành công. Sửa nguyên nhân rồi tạo runtime/run mới; không chạy tiếp cell cuối để ghi đè thất bại. `test_degraded.tar` là protocol lưu sẵn cũ, khác suy thoái sau crop của benchmark mới; không gộp hai loại thành cùng chỉ số.
